In [13]:
from gprofiler import GProfiler
import pandas as pd
import numpy as np
gp = GProfiler(user_agent='ExampleTool', return_dataframe=True)


In [4]:
filename = 'data/final_enr.csv'
columns = ['diseaseId', '#snp', '#border_snp', 'pval_tad', 'pval_border', 'pval_outside', 'is_cancer', 'window_size', 'filter']
df = pd.read_csv(filename, usecols=columns)
print(df.head)

<bound method NDFrame.head of              diseaseId  #snp  #border_snp  pval_tad  pval_border  \
0          EFO_0000094    22           13  0.000000     0.000000   
1          EFO_0000095   130           72  0.000000     0.000000   
2          EFO_0000096    13            6  0.000000     0.000000   
3          EFO_0000174     7            3  0.000000     0.000000   
4          EFO_0000178    37           16  0.000000     0.000000   
...                ...   ...          ...       ...          ...   
71741    Orphanet_1945    11            0       NaN          NaN   
71742  Orphanet_391311    15            5  0.950432     0.172214   
71743     Orphanet_654     7            3  0.969582     0.088117   
71744   Orphanet_93957     3            0       NaN          NaN   
71745   Orphanet_98974     7            2  0.730591     0.294210   

       pval_outside  is_cancer  window_size     filter  
0          1.000000      False            0   nofilter  
1          1.000000      False         

In [5]:
mapped_genes = pd.read_csv('data/combined_diseases_genes.csv', delimiter='\t')

In [9]:
mapped_genes = mapped_genes.rename(columns={'efoID': 'diseaseId'})

In [11]:
combined_df = pd.merge(df, mapped_genes, on='diseaseId', how='inner', indicator=True)

In [14]:
combined_df["Quadrant"] = combined_df.apply(lambda x: "Z+P+" if x.z_score > 2 and x['pval_border'] < 0.05
                          else "Z+P-" if x.z_score > 2
                          else "Z-P+" if x['pval_border'] < 0.05
                          else "Z-P-", axis=1)

In [15]:
combined_df.head()

,diseaseId,#snp,#border_snp,pval_tad,pval_border,pval_outside,is_cancer,window_size,filter,name,z_score,pvalue,p_empirical,mapped-genes,_merge,Quadrant
0,EFO_0000095,130,72,0.0,0.0,1.0,False,0,nofilter,"diffuse large B-cell lymphoma, marginal zone B...",7.770321,3.250000e-06,0.000999,"LRRC34,RREB1,THEMIS,PTPRK,DLEU1,VMP1,RPS6KB1,C...",both,Z+P+
1,EFO_0000195,261,120,0.0,0.0,1.0,False,0,nofilter,"response to antipsychotic drug, metabolic synd...",3.113403,4.440000e-16,0.002997,"GPR139,SNRPEP3,STAM2,APOA5,CATSPER2P1,PPP1R3B,...",both,Z+P+
2,EFO_0000274,111,46,0.0,0.0,1.0,False,0,nofilter,atopic eczema,7.647706,4.370000e-10,0.000999,"EMSY,LINC02757,LINC02571,HLA,B,IL18R1,KRT8P26,...",both,Z+P+
3,EFO_0000275,373,175,0.0,0.0,1.0,False,0,nofilter,atrial fibrillation,4.578832,3.000000e-04,0.002997,"GOSR2,ATXN1,KCNH2,FBXO32,XPO7,GJA5,GJA8,REEP3,...",both,Z+P+
4,EFO_0000305,1135,440,0.0,0.0,1.0,True,0,nofilter,breast carcinoma,4.451704,1.560000e-08,0.000999,"RAD51B,TNIP3,SLC45A1,LASTR,ZMIZ1,GRM7,ADAMTS16...",both,Z+P+


In [30]:
zpp_genes = combined_df[(combined_df["is_cancer"] == True) & (combined_df["Quadrant"] == "Z+P+")]["mapped-genes"].str.split(",").explode().unique().tolist()

result = gp.profile(organism='hsapiens', query=zpp_genes)
result.to_csv("data/go_enrichment_++_cancers.csv", sep='\t')


In [25]:
zpp_genes = combined_df[(combined_df["is_cancer"] == False) & (combined_df["Quadrant"] == "Z+P+")]["mapped-genes"].str.split(",").explode().unique().tolist()

result = gp.profile(organism='hsapiens', query=zpp_genes)

                               name        p_value  term_size
0                   protein binding  9.704646e-239      14865
1                         cytoplasm  1.498622e-227      12394
2             developmental process  2.105073e-222       6478
3  anatomical structure development  7.658328e-217       5924
4  multicellular organismal process  8.609155e-202       7234


In [29]:
result.to_csv("data/go_enrichment_++_noncancers.csv", sep='\t')

In [19]:
zpp_genes = combined_df[combined_df["Quadrant"] == "Z-P+"]["mapped-genes"].str.split(",").explode().unique().tolist()

result = gp.profile(organism='hsapiens', query=zpp_genes)
print(result[['name', 'p_value', 'term_size']].head())

                                 name       p_value  term_size
0  multicellular organism development  6.541054e-83       4658
1                  system development  9.909995e-80       3985
2    multicellular organismal process  8.546810e-79       7234
3    anatomical structure development  8.994351e-78       5924
4               developmental process  1.140722e-76       6478


In [20]:
zpp_genes = combined_df[combined_df["Quadrant"] == "Z+P-"]["mapped-genes"].str.split(",").explode().unique().tolist()

result = gp.profile(organism='hsapiens', query=zpp_genes)
print(result[['name', 'p_value', 'term_size']].head())

                               name        p_value  term_size
0                   protein binding  1.342636e-234      14865
1             developmental process  1.000950e-231       6478
2  anatomical structure development  4.540288e-229       5924
3                         cytoplasm  3.563602e-217      12394
4  multicellular organismal process  3.145034e-213       7234


In [21]:
zpp_genes = combined_df[combined_df["Quadrant"] == "Z-P-"]["mapped-genes"].str.split(",").explode().unique().tolist()

result = gp.profile(organism='hsapiens', query=zpp_genes)
print(result[['name', 'p_value', 'term_size']].head())

                                 name       p_value  term_size
0  multicellular organism development  3.477374e-98       4658
1                  system development  4.827875e-97       3985
2    anatomical structure development  3.462116e-93       5924
3               developmental process  3.071319e-91       6478
4    multicellular organismal process  4.455318e-89       7234
